## Install the packages

In [1]:
%pip install langchain==0.3.27 langchain-community==0.3.27 langchain-huggingface==0.3.1 langchain-core==0.3.74 langchain-chroma==0.2.5 pypdf -q


[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [6]:
%pip install sentence-transformers

  Using cached sentence_transformers-6.0.1-py3-none-any.whl.metadata (20 kB)
  Using cached transformers-5.16.1-py3-none-any.whl.metadata (32 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached safetensors-0.8.0-cp310-abi3-macosx_11_0_arm64.whl.metadata (4.2 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.8/739.8 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 11.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.5/20.5 MB 11.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.2/111.2 MB 11.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 11.6 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 11.3 MB/s eta 0:00:00
Using cached safetensors-0.8.0-cp310-abi3-macosx_11_0_arm64.whl (484 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 818.2/818.2 

## Load Dependencies

In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

## 1) Load the PDF  (one Document per page)




In [2]:
loader = PyPDFLoader("./Belal_Bassem_CV.pdf")

document = loader.load()

In [3]:
document[0]

Document(metadata={'producer': 'pdfTeX-1.40.26', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-07-06T22:05:38+00:00', 'author': '', 'keywords': '', 'moddate': '2026-07-06T22:05:38+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.26 (TeX Live 2024) kpathsea version 6.4.0', 'subject': '', 'title': '', 'trapped': '/False', 'source': './Belal_Bassem_CV.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content='Belal Bassem\n+20-109-896-7689 | bbassem2005@gmail.com | Linkedin | Github\nEducation\nGerman University in Cairo Cairo, Egypt\nBachelor of Engineering in Computer Science and Engineering Sept. 2023 – May 2028\n• Cumulative GPA: 0.97 (Equivalent to 4.0 American Scale)\n• Relevant Courses: Data Structures and Algorithms , Software Engineering , Computer Architecture ,\nObject-Oriented Programming , Linear Algebra , Probability and Statistics , Databases , Discrete Mathematics,\nOperating Systems\nAl Bashaer International School Cairo, Egypt\

## 2) Split into overlapping chunks

In [4]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 1000 , chunk_overlap = 20)
chunks = splitter.split_documents(document)

chunks

[Document(metadata={'producer': 'pdfTeX-1.40.26', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-07-06T22:05:38+00:00', 'author': '', 'keywords': '', 'moddate': '2026-07-06T22:05:38+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.26 (TeX Live 2024) kpathsea version 6.4.0', 'subject': '', 'title': '', 'trapped': '/False', 'source': './Belal_Bassem_CV.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content='Belal Bassem\n+20-109-896-7689 | bbassem2005@gmail.com | Linkedin | Github\nEducation\nGerman University in Cairo Cairo, Egypt\nBachelor of Engineering in Computer Science and Engineering Sept. 2023 – May 2028\n• Cumulative GPA: 0.97 (Equivalent to 4.0 American Scale)\n• Relevant Courses: Data Structures and Algorithms , Software Engineering , Computer Architecture ,\nObject-Oriented Programming , Linear Algebra , Probability and Statistics , Databases , Discrete Mathematics,\nOperating Systems\nAl Bashaer International School Cairo, Egypt

## 3) Embed & Store in Chroma

In [7]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# 3) Create the embedding function
embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')


# 4) Embed all chunks and persist to Chroma
vectordb = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="handbook",
    persist_directory="./chroma_db")

print("Stored", vectordb._collection.count(),
      "vectors")


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 18632.51it/s]


Stored 3 vectors


## 4) Query the Vector Store

In [10]:
# Plug into a RAG chain as a retriever
retriever = vectordb.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
    )

results = retriever.get_relevant_documents(query)
for i, r in enumerate(results):
    print(f"\n chunk : {i} \n")
    print(r.page_content, r.metadata)


 chunk : 0 

{'msip_label_23507802-f8e4-4e38-829c-ac8ea9b241e4_actionid': '033a1253-16ec-424d-94d5-e7380b6ba5b9', 'msip_label_23507802-f8e4-4e38-829c-ac8ea9b241e4_method': 'Privileged', 'creator': 'Microsoft® Word for Microsoft 365', 'msip_label_23507802-f8e4-4e38-829c-ac8ea9b241e4_enabled': 'true', 'page_label': '5', 'page': 4, 'producer': 'Microsoft® Word for Microsoft 365', 'moddate': '2025-05-11T15:47:44+03:00', 'creationdate': '2025-05-11T15:47:44+03:00', 'source': 'FF_PROCEDURE_ENHANCE.pdf', 'author': 'mohamed.elhawary', 'total_pages': 8, 'msip_label_23507802-f8e4-4e38-829c-ac8ea9b241e4_contentbits': '2', 'msip_label_23507802-f8e4-4e38-829c-ac8ea9b241e4_siteid': '6e51e1ad-c54b-4b39-b598-0ffe9ae68fef'}

 chunk : 1 

{'author': 'mohamed.elhawary', 'msip_label_23507802-f8e4-4e38-829c-ac8ea9b241e4_enabled': 'true', 'producer': 'Microsoft® Word for Microsoft 365', 'source': 'FF_PROCEDURE_ENHANCE.pdf', 'msip_label_23507802-f8e4-4e38-829c-ac8ea9b241e4_siteid': '6e51e1ad-c54b-4b39-b598-

## ETL

In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

class PDFToChromaETL:
    def __init__(self, persist_dir="./chroma_db"):
        self.persist_dir = persist_dir
        self.embeddings = HuggingFaceEmbeddings(
            model_name="sentence-transformers/all-MiniLM-L6-v2"
        )

    def extract(self, pdf_path):                 # E
        loader = PyPDFLoader(pdf_path)
        documents = loader.load()
        return documents

    def transform(self, docs):                   # T
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=1000,
            chunk_overlap=20
        )
        chunks = splitter.split_documents(docs)
        return chunks

    def load(self, chunks):                      # L
        db = Chroma.from_documents(
            documents=chunks,
            embedding=self.embeddings,
            persist_directory=self.persist_dir
        )
        return db

    def run(self, pdf_path):
        docs = self.extract(pdf_path)
        chunks = self.transform(docs)
        db = self.load(chunks)
        return db

# --- Run the full ETL job ---
etl = PDFToChromaETL()
db = etl.run("Belal_Bassem_CV.pdf")

query = "What is my role in Jackaroo"
chunks = db.similarity_search(query, k=3)
for i, d in enumerate(chunks):
    print(f"chunk : {i} ")
    print(d.metadata["page"], d.page_content)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7354.04it/s]


chunk : 0 
0 Belal Bassem
+20-109-896-7689 | bbassem2005@gmail.com | Linkedin | Github
Education
German University in Cairo Cairo, Egypt
Bachelor of Engineering in Computer Science and Engineering Sept. 2023 – May 2028
• Cumulative GPA: 0.97 (Equivalent to 4.0 American Scale)
• Relevant Courses: Data Structures and Algorithms , Software Engineering , Computer Architecture ,
Object-Oriented Programming , Linear Algebra , Probability and Statistics , Databases , Discrete Mathematics,
Operating Systems
Al Bashaer International School Cairo, Egypt
International General Certificate of Secondary Education Sept. 2009 – June 2023
• Graduated with a weighted grade of 120%
Projects
Jackaroo
 Github | Java, JavaFX, Object-Oriented Programming, May 2025
• Designed and implemented Jackaroo, a traditional Arab board game in Java, earning an A+ grade.
• Applied object-oriented programming principles with a modular structure using classes and objects.
chunk : 1 
0 • Developed an AI agent using LangCha